In this notebook, I present the results for the analysis of different train sizes.

In [1]:
from sys import path

path.append("../")

from edamame_downstream.utils.result_presentation import get_results_path, present_results

In [2]:
import pandas as pd
from IPython.display import display, HTML

# Widen the notebook output area
display(HTML("<style>.container { width:98% !important; }</style>"))

# Make DataFrame display larger/more detailed
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)   # increase if you want to show more rows
pd.set_option("display.width", 2000)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.colheader_justify", "center")

In [3]:
# TODO: `only_last` does not work as intended. It should consider only the last multirun for each
# dataset, regardless of the others.
all_results = get_results_path(results_paths=['../outputs'], specific_path="multirun_2026-03-03-13-22-48", only_last=False)

# TODO: the error from the average over seed is not computed correctly.
unstructured, structured = present_results(
    all_results,
    
    val_method="lopo",
    remove_xgboost=True,
    remove_chronos_small_from_test=False,
    average_over_seed=True,
    show_tests=False,
    filters={"Channels": "[0, 1, 2]",
             "Model": "LogisticRegression",
             "Sample Scaling": "empty_scaler"}
)


Processing reports: 100%|██████████| 70/70 [00:00<00:00, 205.93it/s]


Seed accuracy_score balanced_accuracy_score   f1_score   matthews_corrcoef precision_score recall_score roc_auc_score
Aggregator         Dataset   Features                    Label Name    Model              Resampling Side  Validation Channels  Feature Scaling Sample Scaling Subsample Train Set                                                                                                                       
MeanTimeAggregator usilaughs amazon/chronos-t5-small     cognitiveload LogisticRegression None       right LOPO       [0, 1, 2] StandardScaler  empty_scaler   0.1                  NaN    0.63 ± 0.08         0.59 ± 0.09       0.69 ± 0.09     0.18 ± 0.18      0.72 ± 0.10   0.72 ± 0.11   0.59 ± 0.09
                                                                                                                                                               0.2                  NaN    0.58 ± 0.08         0.54 ± 0.09       0.61 ± 0.11     0.08 ± 0.17      0.65 ± 0.12   0.65 ± 0.14   0.54 ± 0.09
                                                                                                                                                               0.3                  NaN    0.59 ± 0.09         0.55 ± 0.10       0.64 ± 0.10     0.10 ± 0.19      0.66 ± 0.11   0.67 ± 0.12   0.55 ± 0.10
                                                                                                                                                               0.4                  NaN    0.60 ± 0.09         0.58 ± 0.10       0.62 ± 0.12     0.17 ± 0.19      0.66 ± 0.13   0.63 ± 0.13   0.58 ± 0.10
                                                                                                                                                               0.5                  NaN    0.63 ± 0.09         0.65 ± 0.09       0.62 ± 0.12     0.30 ± 0.18      0.71 ± 0.14   0.60 ± 0.13   0.65 ± 0.09
                                                                                                                                                               0.6                  NaN    0.62 ± 0.09         0.63 ± 0.09       0.60 ± 0.12     0.27 ± 0.18      0.67 ± 0.14   0.60 ± 0.14   0.63 ± 0.09
                                                                                                                                                               0.7                  NaN    0.64 ± 0.08         0.64 ± 0.09       0.68 ± 0.09     0.28 ± 0.18      0.78 ± 0.11   0.65 ± 0.11   0.64 ± 0.09
                                                                                                                                                               0.8                  NaN    0.61 ± 0.07         0.61 ± 0.08       0.63 ± 0.10     0.22 ± 0.16      0.73 ± 0.12   0.62 ± 0.12   0.61 ± 0.08
                                                                                                                                                               0.9                  NaN    0.68 ± 0.07         0.66 ± 0.08       0.69 ± 0.11     0.32 ± 0.15      0.73 ± 0.12   0.72 ± 0.13   0.66 ± 0.08
                                                                                                                                                               1.0                  NaN    0.62 ± 0.08         0.68 ± 0.07       0.57 ± 0.12     0.35 ± 0.14      0.71 ± 0.15   0.52 ± 0.13   0.68 ± 0.07
None               usilaughs AutonLab/MOMENT-1-large     cognitiveload LogisticRegression None       right LOPO       [0, 1, 2] StandardScaler  empty_scaler   0.1                  NaN    0.53 ± 0.09         0.47 ± 0.09       0.61 ± 0.11    -0.07 ± 0.19      0.57 ± 0.10   0.67 ± 0.13   0.47 ± 0.09
                                                                                                                                                               0.2                  NaN    0.61 ± 0.10         0.59 ± 0.10       0.64 ± 0.11     0.18 ± 0.20      0.69 ± 0.12   0.65 ± 0.13   0.59 ± 0.10
        

In [5]:
import ipywidgets as widgets
from IPython.display import display

# Build selector options from the data
_all_features = sorted(structured.reset_index()["Features"].unique().tolist())

feature_selector = widgets.SelectMultiple(
    options=_all_features,
    value=_all_features,
    description="Features:",
    layout=widgets.Layout(height=f"{min(30 * len(_all_features) + 20, 200)}px", width="450px"),
    style={"description_width": "70px"},
)

show_se_toggle = widgets.Checkbox(
    value=True,
    description="Show standard errors",
    indent=False,
)

display(widgets.VBox([feature_selector, show_se_toggle]))


In [7]:
import matplotlib.pyplot as plt
import numpy as np


def plot_trainsize(selected_features, show_se):
    df_plot = structured.reset_index()
    df_plot["Subsample Train Set"] = df_plot["Subsample Train Set"].replace(False, 1)

    # Parse "mean ± sem" strings
    df_plot[["ba_mean", "ba_sem"]] = (
        df_plot["balanced_accuracy_score"]
        .str.split(" ± ", expand=True)
        .astype(float)
    )
    # When no subsampling is applied the value is False -> treat as 1 (full train set)
    df_plot["Subsample Train Set"] = pd.to_numeric(df_plot["Subsample Train Set"], errors="coerce")

    df_plot = df_plot[df_plot["Features"].isin(selected_features)]
    df_plot = df_plot.sort_values(["Features", "Subsample Train Set"])

    fig, ax = plt.subplots(figsize=(10, 6))

    for feature, grp in df_plot.groupby("Features"):
        x = grp["Subsample Train Set"].values
        y = grp["ba_mean"].values
        se = grp["ba_sem"].values
        (line,) = ax.plot(x, y, marker="o", label=feature)
        if show_se:
            ax.fill_between(x, y - se, y + se, alpha=0.2, color=line.get_color())

    ax.set_xlabel("Subsample Train Set", fontsize=13)
    ax.set_ylabel("Balanced Accuracy", fontsize=13)
    ax.set_title("Balanced Accuracy vs. Train Set Size by Feature Type", fontsize=14)
    ax.legend(title="Features", bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=10)
    ax.grid(True, linestyle="--", alpha=0.5)
    plt.tight_layout()
    plt.show()


out = widgets.interactive_output(
    plot_trainsize,
    {"selected_features": feature_selector, "show_se": show_se_toggle},
)
display(out)


Output()

# APSYNC

In [ ]:
# TODO: `only_last` does not work as intended. It should consider only the last multirun for each
# dataset, regardless of the others.
all_results = get_results_path(results_paths=['../outputs'], specific_path="multirun_2026-03-03-13-22-48", only_last=False)

# TODO: the error from the average over seed is not computed correctly.
unstructured, structured = present_results(
    all_results,
    
    val_method="tacv",
    remove_xgboost=True,
    remove_chronos_small_from_test=False,
    average_over_seed=True,
    show_tests=False,
    filters={"Channels": "[0, 1, 2]",
             "Model": "LogisticRegression",
             "Sample Scaling": "empty_scaler"}
)


Processing reports: 100%|██████████| 70/70 [00:00<00:00, 205.93it/s]


Seed accuracy_score balanced_accuracy_score   f1_score   matthews_corrcoef precision_score recall_score roc_auc_score
Aggregator         Dataset   Features                    Label Name    Model              Resampling Side  Validation Channels  Feature Scaling Sample Scaling Subsample Train Set                                                                                                                       
MeanTimeAggregator usilaughs amazon/chronos-t5-small     cognitiveload LogisticRegression None       right LOPO       [0, 1, 2] StandardScaler  empty_scaler   0.1                  NaN    0.63 ± 0.08         0.59 ± 0.09       0.69 ± 0.09     0.18 ± 0.18      0.72 ± 0.10   0.72 ± 0.11   0.59 ± 0.09
                                                                                                                                                               0.2                  NaN    0.58 ± 0.08         0.54 ± 0.09       0.61 ± 0.11     0.08 ± 0.17      0.65 ± 0.12   0.65 ± 0.14   0.54 ± 0.09
                                                                                                                                                               0.3                  NaN    0.59 ± 0.09         0.55 ± 0.10       0.64 ± 0.10     0.10 ± 0.19      0.66 ± 0.11   0.67 ± 0.12   0.55 ± 0.10
                                                                                                                                                               0.4                  NaN    0.60 ± 0.09         0.58 ± 0.10       0.62 ± 0.12     0.17 ± 0.19      0.66 ± 0.13   0.63 ± 0.13   0.58 ± 0.10
                                                                                                                                                               0.5                  NaN    0.63 ± 0.09         0.65 ± 0.09       0.62 ± 0.12     0.30 ± 0.18      0.71 ± 0.14   0.60 ± 0.13   0.65 ± 0.09
                                                                                                                                                               0.6                  NaN    0.62 ± 0.09         0.63 ± 0.09       0.60 ± 0.12     0.27 ± 0.18      0.67 ± 0.14   0.60 ± 0.14   0.63 ± 0.09
                                                                                                                                                               0.7                  NaN    0.64 ± 0.08         0.64 ± 0.09       0.68 ± 0.09     0.28 ± 0.18      0.78 ± 0.11   0.65 ± 0.11   0.64 ± 0.09
                                                                                                                                                               0.8                  NaN    0.61 ± 0.07         0.61 ± 0.08       0.63 ± 0.10     0.22 ± 0.16      0.73 ± 0.12   0.62 ± 0.12   0.61 ± 0.08
                                                                                                                                                               0.9                  NaN    0.68 ± 0.07         0.66 ± 0.08       0.69 ± 0.11     0.32 ± 0.15      0.73 ± 0.12   0.72 ± 0.13   0.66 ± 0.08
                                                                                                                                                               1.0                  NaN    0.62 ± 0.08         0.68 ± 0.07       0.57 ± 0.12     0.35 ± 0.14      0.71 ± 0.15   0.52 ± 0.13   0.68 ± 0.07
None               usilaughs AutonLab/MOMENT-1-large     cognitiveload LogisticRegression None       right LOPO       [0, 1, 2] StandardScaler  empty_scaler   0.1                  NaN    0.53 ± 0.09         0.47 ± 0.09       0.61 ± 0.11    -0.07 ± 0.19      0.57 ± 0.10   0.67 ± 0.13   0.47 ± 0.09
                                                                                                                                                               0.2                  NaN    0.61 ± 0.10         0.59 ± 0.10       0.64 ± 0.11     0.18 ± 0.20      0.69 ± 0.12   0.65 ± 0.13   0.59 ± 0.10
        

In [ ]:
import ipywidgets as widgets
from IPython.display import display

# Build selector options from the data
_all_features = sorted(structured.reset_index()["Features"].unique().tolist())

feature_selector = widgets.SelectMultiple(
    options=_all_features,
    value=_all_features,
    description="Features:",
    layout=widgets.Layout(height=f"{min(30 * len(_all_features) + 20, 200)}px", width="450px"),
    style={"description_width": "70px"},
)

show_se_toggle = widgets.Checkbox(
    value=True,
    description="Show standard errors",
    indent=False,
)

display(widgets.VBox([feature_selector, show_se_toggle]))


In [ ]:
import matplotlib.pyplot as plt
import numpy as np


def plot_trainsize(selected_features, show_se):
    df_plot = structured.reset_index()
    df_plot["Subsample Train Set"] = df_plot["Subsample Train Set"].replace(False, 1)

    # Parse "mean ± sem" strings
    df_plot[["ba_mean", "ba_sem"]] = (
        df_plot["balanced_accuracy_score"]
        .str.split(" ± ", expand=True)
        .astype(float)
    )
    # When no subsampling is applied the value is False -> treat as 1 (full train set)
    df_plot["Subsample Train Set"] = pd.to_numeric(df_plot["Subsample Train Set"], errors="coerce")

    df_plot = df_plot[df_plot["Features"].isin(selected_features)]
    df_plot = df_plot.sort_values(["Features", "Subsample Train Set"])

    fig, ax = plt.subplots(figsize=(10, 6))

    for feature, grp in df_plot.groupby("Features"):
        x = grp["Subsample Train Set"].values
        y = grp["ba_mean"].values
        se = grp["ba_sem"].values
        (line,) = ax.plot(x, y, marker="o", label=feature)
        if show_se:
            ax.fill_between(x, y - se, y + se, alpha=0.2, color=line.get_color())

    ax.set_xlabel("Subsample Train Set", fontsize=13)
    ax.set_ylabel("Balanced Accuracy", fontsize=13)
    ax.set_title("Balanced Accuracy vs. Train Set Size by Feature Type", fontsize=14)
    ax.legend(title="Features", bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=10)
    ax.grid(True, linestyle="--", alpha=0.5)
    plt.tight_layout()
    plt.show()


out = widgets.interactive_output(
    plot_trainsize,
    {"selected_features": feature_selector, "show_se": show_se_toggle},
)
display(out)


Output()